In [ ]:
import pandas as pd

In [ ]:
ewb_26 = pd.read_excel("131026 Base 2026.xlsx", sheet_name=None)

In [ ]:
nombres_hojas = list(ewb_26.keys())
nombres_hojas

In [ ]:
df_final = ewb_26["base"].copy()

In [ ]:
df_final.columns

## Diccionario de normalización 2026
Asocia cada encabezado del 2026 (en **MAYÚSCULAS**, mismo criterio del `column_mapping.yaml`) a una etiqueta estándar (también en **MAYÚSCULAS**) que **ya existe** en la base. No se crean columnas nuevas.

- `DICCIONARIO_2026`: encabezado original → etiqueta estándar.
- `VARIANTES_DUPLICADAS_2026`: encabezados alternos (nombres pegados, de otras hojas) que apuntan a la misma etiqueta.
- Lo que no tiene equivalente (género, fecha de nacimiento, jornada, ubicación, constancia, apellidos) se resolverá después en dbt.

In [ ]:
DICCIONARIO_2026 = {
    # DOCENTE
    "TIPO DE DOCUMENTO": "TIPO_DOCUMENTO",
    "NÚMERO DE DOCUMENTO": "DOCENTE_ID",
    "NOMBRES": "NOMBRES",
    "TELÉFONO": "TELEFONO_CELULAR",
    "CORREO ELECTRÓNICO": "CORREO_ELECTRONICO",

    # IED
    "IED": "COLEGIO_NOMBRE",
    "DANE": "CODIGO_DANE",

    # PERFIL
    "NIVEL DE FORMACION": "NIVEL_FORMACION",
    "CARGO": "CARGO_ROL",
    "TIPO DE NOMBRAMIENTO": "TIPO_NOMBRAMIENTO",
    "GRADOS EN LOS QUE ENSEÑA": "GRADO_DICTA",

    # CURSO
    "GRUPO": "GRUPO_ID",
    "NIVEL": "NIVEL_ASIGNADO",

    # METRICAS
    "ENGLISH SCORE ENTRADA 2026": "ENGLISH_SCORE_ENTRADA_2026",
    "NIVEL MCER ENTRADA 2026": "NIVEL_MCER_ENTRADA_2026",
    "% ASISTENCIA A CLASES": "PORCENTAJE_FINAL",
}

# Encabezados alternos que, tras renombrar, repetirían la etiqueta de arriba.
VARIANTES_DUPLICADAS_2026 = {
    "NIVELDEFORMACIÓN": "NIVEL_FORMACION",
    "TIPODENOMBRAMIENTOENLAINSTI": "TIPO_NOMBRAMIENTO",
    "GRADOSENLOSQUEENSEÑA": "GRADO_DICTA",
    "ESENTRADA2026": "ENGLISH_SCORE_ENTRADA_2026",
    "NIVEL_MCER_ENTRADA_2026": "NIVEL_MCER_ENTRADA_2026",
}

In [ ]:
# Verificación: qué columnas del 2026 quedan mapeadas, duplicadas o sin mapear
cols = [str(c).strip().upper() for c in df_final.columns]

mapeadas = {c: DICCIONARIO_2026[c] for c in cols if c in DICCIONARIO_2026}
variantes = {c: VARIANTES_DUPLICADAS_2026[c] for c in cols if c in VARIANTES_DUPLICADAS_2026}
sin_mapear = [c for c in cols if c not in DICCIONARIO_2026 and c not in VARIANTES_DUPLICADAS_2026]
faltan_en_df = [c for c in DICCIONARIO_2026 if c not in cols]

print(f"Mapeadas: {len(mapeadas)} | Variantes duplicadas: {len(variantes)} | Sin mapear: {len(sin_mapear)}")
print("Del diccionario que NO están en la base:", faltan_en_df)
print("Sin mapear:", sin_mapear)

## Aplicar diccionario y exportar tabla limpia
Renombra con el diccionario (encabezados comparados en MAYÚSCULAS), fusiona las variantes duplicadas (primer valor no nulo) y exporta a `test.xlsx`. Las columnas sin mapear se conservan con su nombre original, también en MAYÚSCULAS.

In [ ]:
def normalizar_2026(df):
    df = df.copy()
    mapa = {**DICCIONARIO_2026, **VARIANTES_DUPLICADAS_2026}
    df.columns = [mapa.get(str(c).strip().upper(), str(c).strip().upper()) for c in df.columns]

    # Fusionar columnas que quedaron con la misma etiqueta (variantes de otras hojas)
    for col in df.columns[df.columns.duplicated()].unique():
        fusion = df.loc[:, df.columns == col].bfill(axis=1).iloc[:, 0]
        df = df.loc[:, df.columns != col]
        df[col] = fusion
    return df

df_clean = normalizar_2026(df_final)
df_clean.to_excel('test.xlsx', index=False)
print(df_clean.shape)
df_clean.head()

In [ ]:
df_clean.columns